In [ ]:
import os
import time
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    ConfusionMatrixDisplay,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

import matplotlib.pyplot as plt

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"

try:
    PROJECT_ROOT = Path(__file__).resolve().parents[1]
except NameError:
    PROJECT_ROOT = Path.cwd().parent

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR = OUTPUTS_DIR / "figures"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_df = pd.read_csv(DATA_PROCESSED_DIR / "train.csv", usecols=["full_text", "label"])
val_df = pd.read_csv(DATA_PROCESSED_DIR / "val.csv", usecols=["full_text", "label"])
test_df = pd.read_csv(DATA_PROCESSED_DIR / "test.csv", usecols=["full_text", "label"])

for df in (train_df, val_df, test_df):
    df["full_text"] = df["full_text"].fillna("").astype(str)
    df["label"] = df["label"].astype(int)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)
print("Train label dist:\n", train_df["label"].value_counts(normalize=True).round(4))

MODEL_NAME = "bert-base-uncased"
MAX_LEN = 256
BATCH_SIZE = 8
EPOCHS = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
GRAD_ACCUM_STEPS = 2
USE_FP16 = bool(torch.cuda.is_available())
EARLY_STOPPING_PATIENCE = 0
ENABLE_GRAD_CHECKPOINTING = False

def load_with_retries(fn, tries=5, wait_sec=3):
    last = None
    for _ in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            time.sleep(wait_sec)
    raise last

def get_tokenizer():
    try:
        return AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True, use_fast=True)
    except Exception:
        return load_with_retries(lambda: AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True))

def get_model():
    def _build():
        m = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
        m.config.id2label = {1: "Fake", 0: "Real"}
        m.config.label2id = {"Fake": 1, "Real": 0}
        return m
    try:
        m = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, local_files_only=True)
        m.config.id2label = {1: "Fake", 0: "Real"}
        m.config.label2id = {"Fake": 1, "Real": 0}
        return m
    except Exception:
        return load_with_retries(_build)

tokenizer = get_tokenizer()

class NewsDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = [str(x) if x is not None else "" for x in texts]
        self.labels = [int(x) for x in labels]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {"text": self.texts[idx], "label": self.labels[idx]}

def make_collate_fn(tok, max_len: int):
    def collate(batch):
        texts = [b["text"] for b in batch]
        labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)
        enc = tok(
            texts,
            truncation=True,
            max_length=int(max_len),
            padding=True,
            return_tensors="pt",
        )
        enc["labels"] = labels
        return enc
    return collate

pin_memory = torch.cuda.is_available()
num_workers = 0

train_loader = DataLoader(
    NewsDataset(train_df["full_text"].tolist(), train_df["label"].tolist()),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
    collate_fn=make_collate_fn(tokenizer, MAX_LEN),
)
val_loader = DataLoader(
    NewsDataset(val_df["full_text"].tolist(), val_df["label"].tolist()),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    collate_fn=make_collate_fn(tokenizer, MAX_LEN),
)
test_loader = DataLoader(
    NewsDataset(test_df["full_text"].tolist(), test_df["label"].tolist()),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    collate_fn=make_collate_fn(tokenizer, MAX_LEN),
)

model = get_model().to(device)

if ENABLE_GRAD_CHECKPOINTING and hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

no_decay = ["bias", "LayerNorm.weight", "LayerNorm.bias"]
param_groups = [
    {
        "params": [p for n, p in model.named_parameters() if p.requires_grad and not any(nd in n for nd in no_decay)],
        "weight_decay": WEIGHT_DECAY,
    },
    {
        "params": [p for n, p in model.named_parameters() if p.requires_grad and any(nd in n for nd in no_decay)],
        "weight_decay": 0.0,
    },
]
optimizer = AdamW(param_groups, lr=LR)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_update_steps = steps_per_epoch * EPOCHS
warmup_steps = int(0.1 * total_update_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_update_steps,
)

scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)

@torch.no_grad()
def run_eval(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0.0
    n_steps = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_loss += float(outputs.loss.item())
        n_steps += 1

        preds = torch.argmax(outputs.logits, dim=1).detach().cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels.detach().cpu().numpy())

    if len(all_preds) == 0:
        return {"loss": 0.0, "accuracy": 0.0, "macro_f1": 0.0, "weighted_f1": 0.0}, np.array([]), np.array([])

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    acc = accuracy_score(all_labels, all_preds)
    _, _, f1_macro, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    _, _, f1_w, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted", zero_division=0)

    return {
        "loss": total_loss / max(1, n_steps),
        "accuracy": acc,
        "macro_f1": f1_macro,
        "weighted_f1": f1_w,
    }, all_labels, all_preds

best_val_f1 = -1.0
best_dir = MODELS_DIR / f"bert_fulltext_trunc{MAX_LEN}"
history = []
patience_left = EARLY_STOPPING_PATIENCE

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    running_loss_raw = 0.0
    step_count = 0
    update_count = 0

    pbar = tqdm(enumerate(train_loader, start=1), total=len(train_loader), desc=f"Epoch {epoch}/{EPOCHS}")
    for step, batch in pbar:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_FP16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss_raw = outputs.loss
            loss = loss_raw / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        running_loss_raw += float(loss_raw.item())
        step_count += 1

        do_step = (step % GRAD_ACCUM_STEPS == 0) or (step == len(train_loader))
        if do_step:
            if USE_FP16:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            if USE_FP16:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            update_count += 1

        if step % 50 == 0 or step == len(train_loader):
            pbar.set_postfix({
                "loss": f"{(running_loss_raw / max(1, step_count)):.4f}",
                "updates": update_count
            })

    val_metrics, _, _ = run_eval(model, val_loader, device)
    train_loss_avg = running_loss_raw / max(1, step_count)

    row = {
        "epoch": epoch,
        "train_loss": train_loss_avg,
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_weighted_f1": val_metrics["weighted_f1"],
    }
    history.append(row)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train_loss={train_loss_avg:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_acc={val_metrics['accuracy']:.4f} | "
        f"val_macro_f1={val_metrics['macro_f1']:.4f}"
    )

    improved = val_metrics["macro_f1"] > best_val_f1
    if improved:
        best_val_f1 = val_metrics["macro_f1"]
        best_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(best_dir)
        tokenizer.save_pretrained(best_dir)
        print("Saved best model to:", str(best_dir))
        patience_left = EARLY_STOPPING_PATIENCE
    else:
        if EARLY_STOPPING_PATIENCE > 0:
            patience_left -= 1
            if patience_left <= 0:
                print(f"Early stopping: no val_macro_f1 improvement for {EARLY_STOPPING_PATIENCE} epoch(s).")
                break

print("Best val macro_f1:", best_val_f1)

history_path = TABLES_DIR / f"bert_history_trunc{MAX_LEN}.csv"
pd.DataFrame(history).to_csv(history_path, index=False)
print("Saved:", str(history_path))

model = AutoModelForSequenceClassification.from_pretrained(best_dir).to(device)
model.eval()

test_metrics, y_test, y_test_pred = run_eval(model, test_loader, device)

print("Test metrics:", test_metrics)
print("\nClassification report (test):")
print(classification_report(
    y_test, y_test_pred,
    labels=[1, 0],
    target_names=["Fake(1)", "Real(0)"],
    digits=4
))

cm = confusion_matrix(y_test, y_test_pred, labels=[1, 0])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fake(1)", "Real(0)"])

plt.figure(figsize=(5.5, 4.5))
ax = plt.gca()
disp.plot(ax=ax, values_format="d", colorbar=True)
plt.title(f"Confusion Matrix (Test) - BERT trunc{MAX_LEN}")
plt.tight_layout()

fig_path = FIGURES_DIR / f"confusion_matrix_bert_trunc{MAX_LEN}.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", str(fig_path))

metrics_path = TABLES_DIR / f"bert_metrics_trunc{MAX_LEN}.csv"
pd.DataFrame([{
    "model": f"BERT_trunc{MAX_LEN}",
    "accuracy": test_metrics["accuracy"],
    "macro_f1": test_metrics["macro_f1"],
    "weighted_f1": test_metrics["weighted_f1"],
    "loss": test_metrics["loss"],
    "best_val_macro_f1": best_val_f1,
}]).to_csv(metrics_path, index=False)
print("Saved:", str(metrics_path))